In [1]:
!pip install fasttext-wheel
!curl -O https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
!pip install sentencepiece
!pip install sentencepiece sacremoses
!pip install emoji

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  1 125.1M   1  1.70M   0      0  1.68M      0   01:14   00:01   01:13  1.68M
  3 125.1M   3  3.90M   0      0  1.94M      0   01:04   00:02   01:02  1.94M
  5 125.1M   5  6.26M   0      0  2.07M      0   01:00   00:03   00:57  2.08M
  6 125.1M   6  8.60M   0      0  2.13M      0   00:58   00:04   00:54  2.13M
  8 125.1M   8 10.90M   0      0  2.17M      0   00:57   00:05   00:52  2.17M
 10 125.1M  10 12.98M   0      0  2.15M      0   00:58   00:06   00:52  2.25M
 12 125.1M  12 15.12M   0      0  2.15M      0   00:58   00:07   00:51  2.23M
 13 125.1M  13 17.27M   0      0  2.15M      0   00:58   00:08   00:50  2.19M
 16 125.1M  16 20.30M   0      0  2.25M      0   00:55   00:09   00:46  2.33M
 18 125.1M  18 23.41M   0      0  2.33M      0   00:53   00:10 

In [2]:
import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, precision_score, recall_score
from transformers import (
    BertTokenizer, BertModel, RobertaTokenizer, RobertaModel,
    MarianMTModel, MarianTokenizer, AutoTokenizer, AutoModel, pipeline
)
import torch
from nltk.stem import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from tqdm import tqdm
import re
import fasttext
import emoji
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

In [3]:
# ── Labels ───────────────────────────────────────────────────────────────────
y_train       = np.load('saved_features/y_train.npy')
y_val         = np.load('saved_features/y_val.npy')
y_train_no_sw = np.load('saved_features/y_train_no_sw.npy')
y_val_no_sw   = np.load('saved_features/y_val_no_sw.npy')
y_train_raw   = np.load('saved_features/y_train_raw.npy')
y_val_raw     = np.load('saved_features/y_val_raw.npy')

# ── Raw ──────────────────────────────────────────────────────────────────────
X_train_raw = np.load('saved_features/X_train_raw.npy', allow_pickle=True)
X_val_raw   = np.load('saved_features/X_val_raw.npy', allow_pickle=True)

# ── BoW ──────────────────────────────────────────────────────────────────────
X_train_bow       = scipy.sparse.load_npz('saved_features/X_train_bow.npz')
X_val_bow         = scipy.sparse.load_npz('saved_features/X_val_bow.npz')
X_train_bow_no_sw = scipy.sparse.load_npz('saved_features/X_train_bow_no_sw.npz')
X_val_bow_no_sw   = scipy.sparse.load_npz('saved_features/X_val_bow_no_sw.npz')

# ── TF-IDF ───────────────────────────────────────────────────────────────────
X_train_tfidf       = scipy.sparse.load_npz('saved_features/X_train_tfidf.npz')
X_val_tfidf         = scipy.sparse.load_npz('saved_features/X_val_tfidf.npz')
X_train_tfidf_no_sw = scipy.sparse.load_npz('saved_features/X_train_tfidf_no_sw.npz')
X_val_tfidf_no_sw   = scipy.sparse.load_npz('saved_features/X_val_tfidf_no_sw.npz')

# ── Word2Vec Skip-gram ────────────────────────────────────────────────────────
X_train_w2v       = np.load('saved_features/X_train_w2v.npy')
X_val_w2v         = np.load('saved_features/X_val_w2v.npy')
X_train_w2v_no_sw = np.load('saved_features/X_train_w2v_no_sw.npy')
X_val_w2v_no_sw   = np.load('saved_features/X_val_w2v_no_sw.npy')

# ── Word2Vec CBOW ─────────────────────────────────────────────────────────────
X_train_w2v_cbow       = np.load('saved_features/X_train_w2v_cbow.npy')
X_val_w2v_cbow         = np.load('saved_features/X_val_w2v_cbow.npy')
X_train_w2v_cbow_no_sw = np.load('saved_features/X_train_w2v_cbow_no_sw.npy')
X_val_w2v_cbow_no_sw   = np.load('saved_features/X_val_w2v_cbow_no_sw.npy')

# ── GloVe ─────────────────────────────────────────────────────────────────────
X_train_glove       = np.load('saved_features/X_train_glove.npy')
X_val_glove         = np.load('saved_features/X_val_glove.npy')
X_train_glove_no_sw = np.load('saved_features/X_train_glove_no_sw.npy')
X_val_glove_no_sw   = np.load('saved_features/X_val_glove_no_sw.npy')

# ── FinBERT ───────────────────────────────────────────────────────────────────
X_train_finbert = np.load('saved_features/X_train_finbert.npy')
X_val_finbert   = np.load('saved_features/X_val_finbert.npy')

# ── RoBERTa ───────────────────────────────────────────────────────────────────
X_train_roberta = np.load('saved_features/X_train_roberta.npy')
X_val_roberta   = np.load('saved_features/X_val_roberta.npy')

# ── BERTweet ──────────────────────────────────────────────────────────────────
X_train_bertweet = np.load('saved_features/X_train_bertweet.npy')
X_val_bertweet   = np.load('saved_features/X_val_bertweet.npy')

print(f"Train size: {len(y_train):,} tweets")
print(f"Validation size: {len(y_val):,} tweets")

Train size: 7,634 tweets
Validation size: 1,909 tweets


In [4]:
def evaluate_model_predictions(y_pred_train, y_pred_val, y_train = y_train, y_val = y_val, show_confusion_matrix = True, show_classification_report = True):
    """
    Evaluate the performance of a classification model on training and validation sets.

    Computes and prints accuracy, F1-score (macro), precision, and recall for both sets.
    Optionally displays the confusion matrix and detailed classification report for the validation set.

    Parameters:
    y_pred_train (array-like): Predicted labels for the training set.
    y_pred_val (array-like): Predicted labels for the validation set.
    y_train (array-like, optional): True labels for the training set.
    y_val (array-like, optional): True labels for the validation set.
    show_confusion_matrix (bool): If True, prints the confusion matrix for the validation set.
    show_classification_report (bool): If True, prints the classification report for the validation set.

    Returns:
    tuple: A tuple containing:
        - train_accuracy (float)
        - train_f1 (float)
        - val_accuracy (float)
        - val_f1 (float)
    """
    
    # to get the accurancy and the macro f1 score of the model on the training set
    train_accuracy = accuracy_score(y_train, y_pred_train)
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    
    # to get the accurancy and the macro f1 score of the model on the validation set
    val_accuracy = accuracy_score(y_val, y_pred_val)
    val_f1 = f1_score(y_val, y_pred_val, average='macro')

    # to get the precision and the recall of the model on the validation set
    val_precision = precision_score(y_val, y_pred_val, average='macro')
    val_recall = recall_score(y_val, y_pred_val, average='macro')

    print(f"Accuracy of train: {train_accuracy:.4f}")
    print(f"F1 Macro (Train): {train_f1:.4f}")
    print(f"Accuracy of val: {val_accuracy:.4f}")
    print(f"\033[1mF1 Macro (Val)\033[0m: {val_f1:.4f}")
    print(f"Precision (Val): {val_precision:.4f}")
    print(f"Recall (Val): {val_recall:.4f}")
    
    # to get the confusion matrix and the classification report of the model on the validation set
    if show_confusion_matrix==True:
        print('\nConfusion Matrix for Validation Data:')    
        print(confusion_matrix(y_val, y_pred_val))

    if show_classification_report==True:
        print('\nClassification Report for Validation Data:')
        print(classification_report(y_val, y_pred_val))

    return val_accuracy, val_f1, val_precision, val_recall

In [5]:
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"

# Load Hugging Face pipeline
hf_classifier = pipeline(
    "text-classification",
    model=MODEL,
    tokenizer=MODEL,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True
)

# Make the predictions
preds_train = hf_classifier(list(X_train_raw))
preds_val = hf_classifier(list(X_val_raw))

# Map model's labels to integers
label_to_int = {
    "LABEL_0": 0,  # negative → Bearish
    "LABEL_1": 2,  # neutral  → Neutral
    "LABEL_2": 1   # positive → Bullish
}
y_pred_train = [label_to_int[pred['label']] for pred in preds_train]
y_pred_val = [label_to_int[pred['label']] for pred in preds_val]

# Evaluation
roberta_accuracy, roberta_f1_macro, roberta_precision, roberta_recall = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_raw, y_val = y_val_raw)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy of train: 0.6865
F1 Macro (Train): 0.5820
Accuracy of val: 0.6794
F1 Macro (Val): 0.5801
Precision (Val): 0.6077
Recall (Val): 0.5649

Confusion Matrix for Validation Data:
[[ 147    2  139]
 [  10  142  233]
 [ 107  121 1008]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.56      0.51      0.53       288
           1       0.54      0.37      0.44       385
           2       0.73      0.82      0.77      1236

    accuracy                           0.68      1909
   macro avg       0.61      0.56      0.58      1909
weighted avg       0.66      0.68      0.67      1909

